<a href="https://colab.research.google.com/github/fnlopezsanz/TP_Lab_Met_Cuantitativos_Inversiones/blob/main/notebook_inversiones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

(Bloque de introducción, borrador)
Pregunta: ¿Que tipos de inversiones le habrían ganado a la inflación en 2025?
(pd: LA IA ayudó a subir este notebook al GitHub)

In [41]:
#importamos librerías
import pandas as pd
import numpy as np
import yfinance as yf

In [42]:
#Descargamos datos
#SPY 500
df_spy = yf.download("SPY", start="2025-01-01", end="2025-12-31", interval="1d") # La IA ayudo a como descargar datos de yfinance
#Plazo UVA
df_uva = pd.read_csv("unidad-de-valor-adquisitivo-uva-20250101-20251231.csv")
#PLazo fijo
df_fijo = pd.read_csv("tasas-de-interes-por-depositos-a-30-dias-de-plazo-en-entidades-financieras-20250101-20251231.csv")
#Inflación
df_inflacion = pd.read_csv("serie_ipc_aperturas.csv", sep=";", encoding='latin1') #Tuvimos ciertos errores al descargar este archivo, Gemini los solucionó poniendo como separador el ;, además de especificar la codificación latin1

/tmp/ipykernel_2117/3075913010.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_spy = yf.download("SPY", start="2025-01-01", end="2025-12-31", interval="1d") # La IA ayudo a como descargar datos de yfinance
[*********************100%***********************]  1 of 1 completed


Empezaremos con la depuración de datos de todos los df, primero viendo su información y la cantidad de valores nulos que tienen (si es que hay), para luego empezar con la limpieza de datos.

*La siguiente celda usó código de IA para reescribir el índice y ordenar las fechas en una columna apate, se usó el siguiente promt a Gemini de Google Colab:
Hola, pensá que eres un analista de datos, necesitas que el df_spy pueda tener un formato que facilite la comparación de la cotización del ETF sobre el tiempo. Por lo tanto, creá un código que reescriba el índice, haciendo que las fechas sean ordenadas en una columna aparte, recordá que las fechas deben estar en formato daytime.*

In [43]:

###COSAS QUE HAY QUE HACER TODAVÍA EN ESTA CELDA!!!
#Ver de aplanar columnas para el df_spy

#Evaluamos el df del SPY 500
print("Forma de df_spy", df_spy.shape)
print()
print("Información de df_spy:")
print(df_spy.info())
#Se puede observar que este df no tiene valores nulos, ya que dentro de cada columna hay 249 valores no nulos
#siendo la misma cantidad que las filas presentes en el df.
#Además, no hay tipos de datos que requieran ser cambiados

#Sin embargo, hay un problema que aparece, no existe una columna que contenga las fechas, por lo que se verá parte del df para ver
#si las fechas están presentes en algún modo
print()
print("Cabeza del df_spy")
print(df_spy.head())
#Si vemos la primera parte del df, se puede observar que las fechas están como índice, para lograr una mejor comparación de la variación del precio con respecto
#al tiempo se tendrá que modificar el índice y ordenar los valores de las fechas en una columna aparte.

#Código de IA
# Reestablecemos el índice para convertir las fechas de Index a una columna común
df_spy = df_spy.reset_index()

# Renombramos la columna del índice para que se llame 'Fecha'
df_spy = df_spy.rename(columns={'Date': 'Fecha'})

# Nos aseguramos de que la columna 'Fecha' esté explícitamente en formato datetime
df_spy['Fecha'] = pd.to_datetime(df_spy['Fecha'])

# Ordenamos los valores de forma ascendente según la fecha
df_spy = df_spy.sort_values(by='Fecha').reset_index(drop=True)

# Evaluamos el resultado final
print("Nueva información de df_spy:")
print(df_spy.info())
print()
print("Primeras filas de df_spy modificado:")
display(df_spy.head())


Forma de df_spy (249, 5)

Información de df_spy:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 249 entries, 2025-01-02 to 2025-12-30
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   (Close, SPY)   249 non-null    float64
 1   (High, SPY)    249 non-null    float64
 2   (Low, SPY)     249 non-null    float64
 3   (Open, SPY)    249 non-null    float64
 4   (Volume, SPY)  249 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 11.7 KB
None

Cabeza del df_spy
Price            Close        High         Low        Open    Volume
Ticker             SPY         SPY         SPY         SPY       SPY
Date                                                                
2025-01-02  573.375732  579.740680  569.315484  578.034214  50204000
2025-01-03  580.544922  581.182362  575.131256  576.210099  37888500
2025-01-06  583.889160  588.145568  582.163061  584.781661  47679400
2025-01-07  577.288940  586.233

Price,Fecha,Close,High,Low,Open,Volume
Ticker,,SPY,SPY,SPY,SPY,SPY
0,2025-01-02,573.375732,579.740680,569.315484,578.034214,50204000
1,2025-01-03,580.544922,581.182362,575.131256,576.210099,37888500
2,2025-01-06,583.889160,588.145568,582.163061,584.781661,47679400
3,2025-01-07,577.288940,586.233222,575.474608,585.909563,60393100
4,2025-01-08,578.132324,579.201350,573.925001,577.357567,47304700


In [44]:
#Evaluamos el df del plazo UVA
print("Forma de df_uva", df_uva.shape)
print()
print("Información de df_uva:")
print(df_uva.info())
#En este caso también se observa que no hay valores nulos dentro del df, sin embargo, la fechas tienen como tipo object,
#esto no es lo más recomendable, por lo que el tipo que tienen los datos de la columna Fecha cambiará al tipo de daytime
#para poder comparar mejor la evolución de la tasa del plazo uva
#en distintos momentos, no hacer este paso podría llevar a errores más adelante en el notebook.
print()
df_uva["Fecha"] = pd.to_datetime(df_uva["Fecha"])
print("Invormación de df_uva con cambios:") #Como se puede ver, ahora los datos de las fechas están en daytime
print(df_uva.info())
print()
print("Cabeza de df_uva:", df_uva.head())

Forma de df_uva (365, 2)

Información de df_uva:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Fecha   365 non-null    object 
 1   Valor   365 non-null    float64
dtypes: float64(1), object(1)
memory usage: 5.8+ KB
None

Invormación de df_uva con cambios:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Fecha   365 non-null    datetime64[ns]
 1   Valor   365 non-null    float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 5.8 KB
None

Cabeza de df_uva:        Fecha    Valor
0 2025-01-01  1301.85
1 2025-01-02  1301.85
2 2025-01-03  1303.84
3 2025-01-04  1304.84
4 2025-01-05  1304.84


In [45]:
#Evaluamos el df del plazo fijo
print("Forma de df_fijo", df_fijo.shape)
print()
print("Información de df_fijo:")
print(df_fijo.info())
#Por las mismas razones explicadas en el caso de df_uva, se cambiará el tipo de datos de la columna Fecha a daytime
df_fijo["Fecha"] = pd.to_datetime(df_fijo["Fecha"])
print()
print("Información de df_fijo con cambios:")
print(df_fijo.info())
print()
print("Cabeza de df_fijo:", df_fijo.head())

Forma de df_fijo (240, 2)

Información de df_fijo:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Fecha   240 non-null    object 
 1   Valor   240 non-null    float64
dtypes: float64(1), object(1)
memory usage: 3.9+ KB
None

Información de df_fijo con cambios:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Fecha   240 non-null    datetime64[ns]
 1   Valor   240 non-null    float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 3.9 KB
None

Cabeza de df_fijo:        Fecha  Valor
0 2025-01-02  31.42
1 2025-01-03  31.80
2 2025-01-06  31.59
3 2025-01-07  31.01
4 2025-01-08  30.32
